# Full Train Network Simulation

This notebook brings together all components for a complete train network simulation:

1. **Trains** moving through routes
2. **Passenger sources** generating arrivals
3. **Sensors** monitoring station queues
4. **Control center** evaluating capacity
5. **Dispatcher** deploying extra trains

We'll run a 30+ minute simulation and analyze performance metrics including:
- Passenger wait times
- Train capacity utilization
- Extra trains deployed
- Queue sizes over time

## Step 1: Setup and Configuration

In [ ]:
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector, MqttPublisher
from simulated_city.agents import (
    Station,
    Train,
    TrainStatus,
    SimulationState,
    TrainAgent,
    PassengerSourceAgent,
    SensorAgent,
    ControlCenterAgent,
    DispatcherAgent,
    Passenger,
)
from datetime import datetime, timedelta
import time
import asyncio

# Load configuration
config = load_config()

# Connect to MQTT
connector = MqttConnector(config.mqtt, client_id_suffix="full-sim")
connector.connect()

if connector.wait_for_connection(timeout=5):
    print("✓ Connected to MQTT broker")
    publisher = MqttPublisher(connector)
else:
    print("✗ Failed to connect to MQTT broker")
    raise ConnectionError("Could not connect to MQTT broker")

print("\nSimulation Configuration:")
print(f"  Train capacity: {config.train_network.train.capacity}")
print(f"  Dispatch threshold: {config.train_network.dispatcher.waiting_threshold}")
print(f"  Peak hours: {[f'{p.start}:00-{p.end}:00' for p in config.train_network.passenger_flow.peak_hours]}")

## Step 2: Initialize Simulation State and Stations

In [ ]:
# Initialize simulation state at peak hour
sim_start_time = datetime.now().replace(hour=18, minute=0, second=0, microsecond=0)
sim_state = SimulationState(current_time=sim_start_time)

print(f"Simulation start time: {sim_start_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Convert config to Station objects
route = []
for station_config in config.train_network.route:
    station = Station(
        name=station_config.name,
        station_type=station_config.type,
        location_lat=station_config.location.lat,
        location_lon=station_config.location.lon,
        exit_percentage=station_config.exit_percentage,
    )
    route.append(station)

entry_stations = [s for s in route if s.station_type == "entry"]
exit_stations = [s for s in route if s.station_type == "exit"]

print(f"\nRoute: {len(route)} stations")
print(f"  Entry stations: {len(entry_stations)}")
print(f"  Exit stations: {len(exit_stations)}")

## Step 3: Create Initial Scheduled Train

We start with one scheduled train that departs according to the regular timetable.

In [ ]:
# Calculate base occupancy
capacity = config.train_network.train.capacity
base_occupancy_percent = config.train_network.train.base_occupancy_percent
base_occupancy_count = int(capacity * base_occupancy_percent / 100)

# Create initial train
initial_train = Train(
    id="train-scheduled-001",
    capacity=capacity,
    current_station_index=0,
    current_station_name=route[0].name,
    base_occupancy_count=base_occupancy_count,
    status=TrainStatus.AT_STATION,
)

sim_state.add_train(initial_train)

# Create train agent
initial_train_agent = TrainAgent(
    train=initial_train,
    route=route,
    mqtt_publisher=publisher,
    mqtt_base_topic=config.train_network.mqtt_base_topic,
)

print(f"✓ Created scheduled train: {initial_train.id}")
print(f"  Capacity: {initial_train.capacity}")
print(f"  Base occupancy: {initial_train.base_occupancy_count}")
print(f"  Available: {initial_train.available_capacity}")

## Step 4: Create Passenger Source Agents

In [ ]:
# Create passenger source agents for each entry station
passenger_sources = []

for entry_station in entry_stations:
    source = PassengerSourceAgent(
        station=entry_station,
        exit_stations=exit_stations,
        passenger_flow_config=config.train_network.passenger_flow,
        mqtt_publisher=publisher,
        mqtt_base_topic=config.train_network.mqtt_base_topic,
    )
    passenger_sources.append(source)

print(f"✓ Created {len(passenger_sources)} passenger source agents")

## Step 5: Create Sensor Agents

In [ ]:
# Create sensor agents
sensor_agents = []

for entry_station in entry_stations:
    sensor = SensorAgent(
        station=entry_station,
        mqtt_publisher=publisher,
        mqtt_base_topic=config.train_network.mqtt_base_topic,
    )
    sensor_agents.append(sensor)

print(f"✓ Created {len(sensor_agents)} sensor agents")

## Step 6: Create Control Center and Dispatcher

In [ ]:
# Create control center
control_center = ControlCenterAgent(
    dispatcher_config=config.train_network.dispatcher,
    mqtt_connector=connector,
    mqtt_publisher=publisher,
    mqtt_base_topic=config.train_network.mqtt_base_topic,
)

# Create dispatcher
dispatcher = DispatcherAgent(
    train_config=config.train_network.train,
    route=route,
    mqtt_connector=connector,
    mqtt_publisher=publisher,
    mqtt_base_topic=config.train_network.mqtt_base_topic,
)

print("✓ Created control center and dispatcher")

## Step 7: Run Simulation

We'll simulate 30 minutes of operation with the following cycle:
1. Generate passengers at entry stations
2. Sensors observe and publish queue sizes
3. Control center evaluates and requests extra trains if needed
4. Dispatcher deploys extra trains
5. Trains pick up and drop off passengers

**Note**: This is a simplified synchronous simulation. In production, agents would run asynchronously.

In [ ]:
# Simulation parameters
SIMULATION_MINUTES = 30
MINUTES_PER_STEP = 5  # Each step represents 5 minutes
STEPS = SIMULATION_MINUTES // MINUTES_PER_STEP

# Tracking for visualization
simulation_log = []
dispatch_events = []

print(f"Starting {SIMULATION_MINUTES}-minute simulation...")
print(f"  Steps: {STEPS} (each {MINUTES_PER_STEP} minutes)\n")

for step in range(STEPS):
    # Update simulation time
    sim_state.current_time = sim_start_time + timedelta(minutes=step * MINUTES_PER_STEP)
    current_hour = sim_state.current_time.hour
    
    print(f"=== Step {step + 1}/{STEPS}: {sim_state.current_time.strftime('%H:%M')} ===")
    
    # 1. Generate passengers at entry stations
    for source in passenger_sources:
        rate = source.get_arrival_rate(current_hour)
        # Scale rate for 5-minute interval (rate is per 10 minutes)
        passengers_this_step = int(rate * (MINUTES_PER_STEP / 10))
        
        if passengers_this_step > 0:
            new_passengers = source.generate_passengers(passengers_this_step)
            queue = sim_state.get_station_queue(source.station.name)
            queue.add_passengers(new_passengers)
            sim_state.total_passengers_generated += len(new_passengers)
            source.publish_arrival(len(new_passengers), queue.count)
    
    print(f"  Passengers generated: +{passengers_this_step * len(entry_stations)}")
    print(f"  Total waiting: {sim_state.total_waiting_passengers}")
    
    # 2. Sensors publish observations
    for sensor in sensor_agents:
        queue = sim_state.get_station_queue(sensor.station.name)
        waiting_count = sensor.count_waiting(queue)
        avg_wait_time = queue.average_wait_time_seconds
        sensor.publish_observation(waiting_count, avg_wait_time)
    
    # 3. Control center evaluates and requests extra trains
    for entry_station in entry_stations:
        queue = sim_state.get_station_queue(entry_station.name)
        if control_center.evaluate_threshold(entry_station.name, queue.count):
            print(f"  ⚠️ {entry_station.name}: {queue.count} waiting (threshold: {config.train_network.dispatcher.waiting_threshold})")
            control_center.request_extra_train(entry_station.name, queue.count)
            
            # Dispatcher deploys extra train
            extra_train_agent = dispatcher.deploy_train(sim_state)
            print(f"  ➜ Extra train deployed: {extra_train_agent.train.id}")
            
            dispatch_events.append({
                "time": sim_state.current_time,
                "station": entry_station.name,
                "waiting_count": queue.count,
                "train_id": extra_train_agent.train.id,
            })
    
    # 4. Simulate scheduled train passing through one station per step
    if initial_train.current_station_index < len(route):
        station = route[initial_train.current_station_index]
        initial_train.current_station_name = station.name
        initial_train.status = TrainStatus.AT_STATION
        
        if station.station_type == "entry":
            # Pick up passengers
            initial_train.status = TrainStatus.BOARDING
            queue = sim_state.get_station_queue(station.name)
            boarded = initial_train_agent.pick_up_passengers(queue)
            sim_state.total_passengers_boarded += boarded
            
            if boarded > 0:
                print(f"  {initial_train.id} at {station.name}: picked up {boarded} passengers")
        
        elif station.station_type == "exit":
            # Drop off passengers
            initial_train.status = TrainStatus.ALIGHTING
            alighted = initial_train_agent.drop_off_passengers(station)
            sim_state.total_passengers_alighted += alighted
            
            if alighted > 0:
                print(f"  {initial_train.id} at {station.name}: dropped off {alighted} passengers")
        
        initial_train_agent.publish_status()
        initial_train.current_station_index += 1
    
    # Log state for visualization
    simulation_log.append({
        "time": sim_state.current_time,
        "step": step,
        "total_waiting": sim_state.total_waiting_passengers,
        "total_generated": sim_state.total_passengers_generated,
        "total_boarded": sim_state.total_passengers_boarded,
        "total_alighted": sim_state.total_passengers_alighted,
        "active_trains": sim_state.active_train_count,
        "extra_trains_deployed": sim_state.extra_trains_deployed,
        "avg_train_capacity": sim_state.average_train_capacity,
        "station_queues": {name: queue.count for name, queue in sim_state.station_queues.items()},
    })
    
    print(f"  Active trains: {sim_state.active_train_count}, Extra deployed: {sim_state.extra_trains_deployed}")
    print()
    time.sleep(0.5)  # Brief pause for readability

print("\n" + "="*60)
print("Simulation complete!")
print("="*60)

## Step 8: Analyze Performance Metrics

In [ ]:
print("\n📊 SIMULATION SUMMARY\n")
print("Passengers:")
print(f"  Generated: {sim_state.total_passengers_generated}")
print(f"  Boarded: {sim_state.total_passengers_boarded}")
print(f"  Alighted: {sim_state.total_passengers_alighted}")
print(f"  Still waiting: {sim_state.total_waiting_passengers}")

boarding_rate = (sim_state.total_passengers_boarded / sim_state.total_passengers_generated * 100) if sim_state.total_passengers_generated > 0 else 0
print(f"  Boarding rate: {boarding_rate:.1f}%")

print(f"\nTrains:")
print(f"  Active trains: {sim_state.active_train_count}")
print(f"  Extra trains deployed: {sim_state.extra_trains_deployed}")
print(f"  Avg capacity utilization: {sim_state.average_train_capacity:.1f}%")

print(f"\nDispatch Events: {len(dispatch_events)}")
if dispatch_events:
    for i, event in enumerate(dispatch_events, 1):
        print(f"  {i}. {event['time'].strftime('%H:%M')} - {event['station']} ({event['waiting_count']} waiting)")

## Step 9: Visualize Queue Sizes Over Time

Let's create a simple text-based visualization of how queues evolved.

In [ ]:
print("\n📈 QUEUE SIZES OVER TIME\n")

# Print header
print(f"{'Time':<8}", end="")
for station in entry_stations:
    print(f"{station.name[:15]:<16}", end="")
print("Total")
print("-" * (8 + 16 * len(entry_stations) + 8))

# Print data
for log_entry in simulation_log:
    time_str = log_entry['time'].strftime('%H:%M')
    print(f"{time_str:<8}", end="")
    
    for station in entry_stations:
        count = log_entry['station_queues'].get(station.name, 0)
        
        # Create visual bar
        bar_length = min(count // 20, 10)  # Scale down for display
        bar = "█" * bar_length
        print(f"{count:>3} {bar:<10}  ", end="")
    
    print(f"{log_entry['total_waiting']:>4}")

## Step 10: Station-by-Station Summary

In [ ]:
print("\n📍 FINAL STATION STATUS\n")

for station in entry_stations:
    queue = sim_state.get_station_queue(station.name)
    print(f"{station.name}:")
    print(f"  Waiting: {queue.count} passengers")
    print(f"  Avg wait time: {queue.average_wait_time_seconds / 60:.1f} minutes")
    
    # Check if over threshold
    threshold = config.train_network.dispatcher.waiting_threshold
    if queue.count > threshold:
        print(f"  ⚠️ OVER THRESHOLD ({threshold})")
    print()

## Step 11: Export Data for Analysis

Save simulation data for further analysis or visualization in other tools.

In [ ]:
import json

# Prepare data for export
export_data = {
    "simulation_config": {
        "start_time": sim_start_time.isoformat(),
        "duration_minutes": SIMULATION_MINUTES,
        "train_capacity": capacity,
        "dispatch_threshold": config.train_network.dispatcher.waiting_threshold,
    },
    "summary": {
        "passengers_generated": sim_state.total_passengers_generated,
        "passengers_boarded": sim_state.total_passengers_boarded,
        "passengers_alighted": sim_state.total_passengers_alighted,
        "passengers_waiting": sim_state.total_waiting_passengers,
        "extra_trains_deployed": sim_state.extra_trains_deployed,
        "boarding_rate_percent": boarding_rate,
    },
    "dispatch_events": [
        {
            "time": event["time"].isoformat(),
            "station": event["station"],
            "waiting_count": event["waiting_count"],
            "train_id": event["train_id"],
        }
        for event in dispatch_events
    ],
    "timeline": [
        {
            "time": log["time"].isoformat(),
            "step": log["step"],
            "total_waiting": log["total_waiting"],
            "active_trains": log["active_trains"],
            "station_queues": log["station_queues"],
        }
        for log in simulation_log
    ],
}

# Display sample
print("Sample export data:")
print(json.dumps(export_data["summary"], indent=2))

# Optionally save to file
# with open('train_simulation_results.json', 'w') as f:
#     json.dump(export_data, f, indent=2)
# print("\n✓ Data saved to train_simulation_results.json")

## Cleanup

In [ ]:
# Disconnect from MQTT
if connector.client and connector.client.is_connected():
    connector.disconnect()
    print("✓ Disconnected from MQTT broker")

## Exercise: Experiment with Simulation Parameters

Try these experiments to understand system behavior:

### 1. Change Peak Hours
Modify `config.yaml` to simulate different peak patterns:
```yaml
peak_hours:
  - start: 7
    end: 9
    passengers_per_10min: 250
  - start: 17
    end: 19
    passengers_per_10min: 350
```

### 2. Adjust Dispatch Threshold
Try different threshold values to see their effect on:
- Number of extra trains deployed
- Passenger wait times
- System efficiency

Lower threshold (200): More trains, shorter waits, higher cost
Higher threshold (300): Fewer trains, longer waits, lower cost

### 3. Simulate a Special Event
Add a surge of passengers at one station:
```python
# Before running simulation, add event passengers
event_station = entry_stations[0]
event_queue = sim_state.get_station_queue(event_station.name)

event_passengers = [
    Passenger(f"event-{i}", event_station.name, exit_stations[0].name, sim_start_time)
    for i in range(400)  # Concert crowd
]
event_queue.add_passengers(event_passengers)
sim_state.total_passengers_generated += len(event_passengers)

print(f"Event simulation: +{len(event_passengers)} passengers at {event_station.name}")
```

### 4. Extend Simulation Duration
Change `SIMULATION_MINUTES` to 60 or 120 to observe:
- Long-term queue behavior
- Multiple dispatch cycles
- System equilibrium

## Key Insights

From this simulation, you should observe:

1. **Peak hour pressure**: During peak hours (18:00-20:00), passenger arrival rate significantly exceeds train capacity
2. **Threshold behavior**: When queues exceed the threshold, the system automatically deploys extra trains
3. **Capacity utilization**: Trains typically operate near capacity during peak hours
4. **Wait times**: Passengers experience longer wait times at stations that reach threshold
5. **System dynamics**: The interaction between passenger generation, train movement, and dispatch decisions creates complex system behavior

## Next Steps

To enhance this simulation:
1. Add map visualization showing train positions and queue sizes
2. Implement async agent loops for realistic concurrency
3. Add weather or incident effects that delay trains
4. Create a real-time dashboard with live metrics
5. Optimize dispatch strategy using machine learning